In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DATA_DIR = Path(
    "/content/drive/MyDrive/Dataset/Training_Dataset"
)

print("Dataset exists:", DATA_DIR.exists())
print("Class folders:", [
    folder.name
    for folder in DATA_DIR.iterdir()
    if folder.is_dir()
])

In [ ]:
from torchvision import datasets

dataset = datasets.ImageFolder(str(DATA_DIR))

print("Total images:", len(dataset))
print("Class mapping:", dataset.class_to_idx)
print("First image path:", dataset.samples[0][0])
print("First image label:", dataset.samples[0][1])

In [ ]:
from collections import Counter

class_counts = Counter(dataset.targets)

print("Images per class:\n")

for class_name, class_idx in dataset.class_to_idx.items():
    count = class_counts[class_idx]
    percentage = 100 * count / len(dataset)

    print(
        f"{class_name}: {count} images "
        f"({percentage:.2f}%)"
    )

In [ ]:
# ============================================================
# 1. IMPORT LIBRARIES AND SETUP
# ============================================================

from pathlib import Path
import copy
import random
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import EfficientNet_B0_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
    matthews_corrcoef,
    cohen_kappa_score,
    log_loss,
    hamming_loss,
    jaccard_score,
    top_k_accuracy_score
)
from sklearn.preprocessing import label_binarize

from IPython.display import display

from google.colab import drive
drive.mount("/content/drive")

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Dataset directory
DATA_DIR = Path(
    "/content/drive/MyDrive/Dataset/Training_Dataset"
)

assert DATA_DIR.exists(), f"Dataset not found: {DATA_DIR}"

# Device
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("Dataset exists:", DATA_DIR.exists())

In [ ]:
# ============================================================
# 2. LOAD DATASET
# ============================================================

# ImageNet normalization for pretrained EfficientNet-B0
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

IMAGE_SIZE = 224

# Training transformations
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10
    ),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# Validation and testing transformations
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# Separate dataset objects allow different transforms
train_dataset_full = datasets.ImageFolder(
    root=str(DATA_DIR),
    transform=train_transform
)

eval_dataset_full = datasets.ImageFolder(
    root=str(DATA_DIR),
    transform=eval_transform
)

class_names = train_dataset_full.classes
num_classes = len(class_names)
targets = np.array(train_dataset_full.targets)

print("Classes:", class_names)
print("Number of classes:", num_classes)
print("Total images:", len(train_dataset_full))

assert num_classes == 4, (
    f"Expected 4 classes, found {num_classes}"
)

# Class distribution
class_counts = pd.Series(targets).value_counts().sort_index()

distribution_df = pd.DataFrame({
    "Class": class_names,
    "Images": [
        int(class_counts.get(i, 0))
        for i in range(num_classes)
    ]
})

distribution_df["Percentage"] = (
    distribution_df["Images"]
    / len(train_dataset_full) * 100
).round(2)

display(distribution_df)

plt.figure(figsize=(8, 5))
sns.barplot(
    data=distribution_df,
    x="Class",
    y="Images",
    hue="Class",
    palette="viridis",
    legend=False
)
plt.title("Dataset Class Distribution")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 3. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

all_indices = np.arange(len(targets))

# First: 80% training, 20% temporary
train_indices, temp_indices = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=SEED,
    stratify=targets
)

# Second: split temporary into 10% validation and 10% testing
val_indices, test_indices = train_test_split(
    temp_indices,
    test_size=0.50,
    random_state=SEED,
    stratify=targets[temp_indices]
)

train_dataset = Subset(
    train_dataset_full,
    train_indices.tolist()
)

val_dataset = Subset(
    eval_dataset_full,
    val_indices.tolist()
)

test_dataset = Subset(
    eval_dataset_full,
    test_indices.tolist()
)

print(f"Training images:   {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")
print(f"Testing images:    {len(test_dataset)}")

print(
    f"Training percentage: "
    f"{100 * len(train_dataset) / len(targets):.1f}%"
)
print(
    f"Validation percentage: "
    f"{100 * len(val_dataset) / len(targets):.1f}%"
)
print(
    f"Testing percentage: "
    f"{100 * len(test_dataset) / len(targets):.1f}%"
)

# Check the distribution in each split
split_distribution = []

for split_name, indices in [
    ("Train", train_indices),
    ("Validation", val_indices),
    ("Test", test_indices)
]:
    counts = np.bincount(
        targets[indices],
        minlength=num_classes
    )

    for i, class_name in enumerate(class_names):
        split_distribution.append({
            "Split": split_name,
            "Class": class_name,
            "Images": int(counts[i])
        })

split_df = pd.DataFrame(split_distribution)

display(
    split_df.pivot(
        index="Class",
        columns="Split",
        values="Images"
    )
)

# Data loaders
BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

print("Data loaders created successfully.")

In [ ]:
# ============================================================
# 4. VISUALIZE SAMPLE IMAGES
# ============================================================

def denormalize_image(tensor):
    mean = torch.tensor(
        IMAGENET_MEAN
    ).view(3, 1, 1)

    std = torch.tensor(
        IMAGENET_STD
    ).view(3, 1, 1)

    image = tensor.cpu() * std + mean

    return image.clamp(0, 1)


images, labels = next(iter(train_loader))

plt.figure(figsize=(12, 8))

for i in range(min(12, len(images))):
    plt.subplot(3, 4, i + 1)

    image = denormalize_image(images[i])

    plt.imshow(image.permute(1, 2, 0))
    plt.title(class_names[labels[i].item()])
    plt.axis("off")

plt.suptitle("Sample Training Images")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 5. EFFICIENTNET-B0 MODEL
# ============================================================

weights = EfficientNet_B0_Weights.DEFAULT

model = models.efficientnet_b0(weights=weights)

# Replace the original 1000-class classifier
in_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(
    in_features,
    num_classes
)

model = model.to(DEVICE)

# Loss function
criterion = nn.CrossEntropyLoss()

# AdamW optimizer
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Reduce LR when validation loss stops improving
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

print(model.classifier)
print("EfficientNet-B0 loaded successfully.")
print("Trainable parameters:", sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
))

In [ ]:
# ============================================================
# 6. TRAINING LOOP
# ============================================================

EPOCHS = 15
PATIENCE = 5

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/EfficientNet_B0_Results"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = OUTPUT_DIR / "best_efficientnet_b0.pth"

history = {
    "train_loss": [],
    "val_loss": [],
    "train_accuracy": [],
    "val_accuracy": [],
    "learning_rate": []
}

best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0

start_time = time.time()

for epoch in range(EPOCHS):

    # ---------------- Training ----------------
    model.train()

    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)

        train_loss_sum += loss.item() * batch_size

        predictions = outputs.argmax(dim=1)

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += batch_size

    train_loss = train_loss_sum / train_total
    train_accuracy = train_correct / train_total

    # ---------------- Validation ----------------
    model.eval()

    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            batch_size = labels.size(0)

            val_loss_sum += loss.item() * batch_size

            predictions = outputs.argmax(dim=1)

            val_correct += (
                predictions == labels
            ).sum().item()

            val_total += batch_size

    val_loss = val_loss_sum / val_total
    val_accuracy = val_correct / val_total

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_accuracy"].append(val_accuracy)
    history["learning_rate"].append(current_lr)

    print(
        f"Epoch [{epoch + 1:02d}/{EPOCHS}] "
        f"| Train Loss: {train_loss:.4f} "
        f"| Train Acc: {train_accuracy:.4f} "
        f"| Val Loss: {val_loss:.4f} "
        f"| Val Acc: {val_accuracy:.4f} "
        f"| LR: {current_lr:.2e}"
    )

    # Save best checkpoint
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch + 1
        epochs_without_improvement = 0

        torch.save({
            "epoch": best_epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "class_names": class_names,
            "best_val_loss": best_val_loss,
            "image_size": IMAGE_SIZE
        }, BEST_MODEL_PATH)

        print("  -> Best model saved.")

    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping triggered.")
        break

training_time = time.time() - start_time

print("\nTraining completed.")
print(f"Best epoch: {best_epoch}")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Training time: {training_time / 60:.2f} minutes")
print("Best model:", BEST_MODEL_PATH)

In [ ]:
# ============================================================
# 7. TRAINING CURVES
# ============================================================

epochs_range = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(
    epochs_range,
    history["train_loss"],
    label="Training Loss",
    marker="o"
)

axes[0].plot(
    epochs_range,
    history["val_loss"],
    label="Validation Loss",
    marker="o"
)

axes[0].set_title("Training vs Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy curve
axes[1].plot(
    epochs_range,
    np.array(history["train_accuracy"]) * 100,
    label="Training Accuracy",
    marker="o"
)

axes[1].plot(
    epochs_range,
    np.array(history["val_accuracy"]) * 100,
    label="Validation Accuracy",
    marker="o"
)

axes[1].set_title("Training vs Validation Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "training_curves.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# Save training history
pd.DataFrame(history).to_csv(
    OUTPUT_DIR / "training_history.csv",
    index=False
)

In [ ]:
# ============================================================
# 8. LOAD BEST MODEL AND PREDICT TEST SET
# ============================================================

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=True
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

y_true_list = []
y_pred_list = []
y_prob_list = []
test_loss_sum = 0.0
test_total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        outputs = model(images)

        loss = criterion(outputs, labels)

        probabilities = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)

        batch_size = labels.size(0)

        test_loss_sum += loss.item() * batch_size
        test_total += batch_size

        y_true_list.extend(labels.cpu().numpy())
        y_pred_list.extend(predictions.cpu().numpy())
        y_prob_list.extend(probabilities.cpu().numpy())

y_true = np.asarray(y_true_list, dtype=int)
y_pred = np.asarray(y_pred_list, dtype=int)
y_prob = np.asarray(y_prob_list, dtype=float)

test_loss = test_loss_sum / test_total

print("Test samples:", len(y_true))
print("Test loss:", round(test_loss, 4))
print("Probability array shape:", y_prob.shape)
print("Prediction array shape:", y_pred.shape)

assert y_prob.shape == (len(y_true), num_classes)

In [ ]:
# ============================================================
# 9. OVERALL EVALUATION METRICS
# ============================================================

labels_all = np.arange(num_classes)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=labels_all
)

# Per-class true positives, false positives,
# false negatives, and true negatives
TP = np.diag(cm).astype(float)
FP = cm.sum(axis=0).astype(float) - TP
FN = cm.sum(axis=1).astype(float) - TP
TN = cm.sum().astype(float) - TP - FP - FN

epsilon = 1e-12

per_class_sensitivity = TP / (TP + FN + epsilon)
per_class_specificity = TN / (TN + FP + epsilon)
per_class_npv = TN / (TN + FN + epsilon)
per_class_fpr = FP / (FP + TN + epsilon)
per_class_fnr = FN / (FN + TP + epsilon)

# ROC-AUC
y_true_bin = label_binarize(
    y_true,
    classes=labels_all
)

roc_auc_macro_ovr = roc_auc_score(
    y_true,
    y_prob,
    labels=labels_all,
    multi_class="ovr",
    average="macro"
)

roc_auc_weighted_ovr = roc_auc_score(
    y_true,
    y_prob,
    labels=labels_all,
    multi_class="ovr",
    average="weighted"
)

roc_auc_macro_ovo = roc_auc_score(
    y_true,
    y_prob,
    labels=labels_all,
    multi_class="ovo",
    average="macro"
)

# Average precision
ap_per_class = average_precision_score(
    y_true_bin,
    y_prob,
    average=None
)

# General metrics
metrics = {
    "Test Loss": test_loss,

    "Accuracy": accuracy_score(y_true, y_pred),

    "Balanced Accuracy": balanced_accuracy_score(
        y_true, y_pred
    ),

    "Precision (Macro)": precision_score(
        y_true, y_pred,
        average="macro",
        zero_division=0
    ),

    "Precision (Micro)": precision_score(
        y_true, y_pred,
        average="micro",
        zero_division=0
    ),

    "Precision (Weighted)": precision_score(
        y_true, y_pred,
        average="weighted",
        zero_division=0
    ),

    "Recall / Sensitivity (Macro)": recall_score(
        y_true, y_pred,
        average="macro",
        zero_division=0
    ),

    "Recall (Micro)": recall_score(
        y_true, y_pred,
        average="micro",
        zero_division=0
    ),

    "Recall (Weighted)": recall_score(
        y_true, y_pred,
        average="weighted",
        zero_division=0
    ),

    "F1 Score (Macro)": f1_score(
        y_true, y_pred,
        average="macro",
        zero_division=0
    ),

    "F1 Score (Micro)": f1_score(
        y_true, y_pred,
        average="micro",
        zero_division=0
    ),

    "F1 Score (Weighted)": f1_score(
        y_true, y_pred,
        average="weighted",
        zero_division=0
    ),

    "Specificity (Macro)": np.mean(
        per_class_specificity
    ),

    "Negative Predictive Value (Macro)": np.mean(
        per_class_npv
    ),

    "False Positive Rate (Macro)": np.mean(
        per_class_fpr
    ),

    "False Negative Rate (Macro)": np.mean(
        per_class_fnr
    ),

    "ROC-AUC (Macro OvR)": roc_auc_macro_ovr,

    "ROC-AUC (Weighted OvR)": roc_auc_weighted_ovr,

    "ROC-AUC (Macro OvO)": roc_auc_macro_ovo,

    "Average Precision (Macro)": np.mean(ap_per_class),

    "Matthews Correlation Coefficient": matthews_corrcoef(
        y_true, y_pred
    ),

    "Cohen's Kappa": cohen_kappa_score(
        y_true, y_pred
    ),

    "Jaccard Score (Macro)": jaccard_score(
        y_true, y_pred,
        average="macro",
        zero_division=0
    ),

    "Log Loss": log_loss(
        y_true, y_prob,
        labels=labels_all
    ),

    "Hamming Loss": hamming_loss(
        y_true, y_pred
    ),

    "Top-2 Accuracy": top_k_accuracy_score(
        y_true,
        y_prob,
        k=2,
        labels=labels_all
    )
}

metrics_df = pd.DataFrame({
    "Metric": list(metrics.keys()),
    "Value": list(metrics.values())
})

metrics_df["Value"] = metrics_df["Value"].astype(float)

display(
    metrics_df.style.format({"Value": "{:.4f}"})
)

metrics_df.to_csv(
    OUTPUT_DIR / "overall_metrics.csv",
    index=False
)

with open(OUTPUT_DIR / "overall_metrics.json", "w") as f:
    json.dump(
        {k: float(v) for k, v in metrics.items()},
        f,
        indent=4
    )

In [ ]:
# ============================================================
# 10. CLASSIFICATION REPORT
# ============================================================

report_dict = classification_report(
    y_true,
    y_pred,
    labels=labels_all,
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report_dict).T

print("Classification Report:\n")

print(
    classification_report(
        y_true,
        y_pred,
        labels=labels_all,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

report_df.to_csv(
    OUTPUT_DIR / "classification_report.csv"
)

# Per-class sensitivity and specificity
per_class_df = pd.DataFrame({
    "Class": class_names,
    "Support": cm.sum(axis=1),
    "TP": TP.astype(int),
    "FP": FP.astype(int),
    "FN": FN.astype(int),
    "TN": TN.astype(int),
    "Precision": precision_score(
        y_true,
        y_pred,
        labels=labels_all,
        average=None,
        zero_division=0
    ),
    "Sensitivity / Recall": per_class_sensitivity,
    "Specificity": per_class_specificity,
    "Negative Predictive Value": per_class_npv,
    "False Positive Rate": per_class_fpr,
    "False Negative Rate": per_class_fnr,
    "F1 Score": f1_score(
        y_true,
        y_pred,
        labels=labels_all,
        average=None,
        zero_division=0
    ),
    "ROC-AUC (OvR)": roc_auc_score(
        y_true_bin,
        y_prob,
        average=None
    ),
    "Average Precision": ap_per_class
})

display(
    per_class_df.style.format({
        "Precision": "{:.4f}",
        "Sensitivity / Recall": "{:.4f}",
        "Specificity": "{:.4f}",
        "Negative Predictive Value": "{:.4f}",
        "False Positive Rate": "{:.4f}",
        "False Negative Rate": "{:.4f}",
        "F1 Score": "{:.4f}",
        "ROC-AUC (OvR)": "{:.4f}",
        "Average Precision": "{:.4f}"
    })
)

per_class_df.to_csv(
    OUTPUT_DIR / "per_class_metrics.csv",
    index=False
)

In [ ]:
# ============================================================
# 11. CONFUSION MATRIX
# ============================================================

plt.figure(figsize=(9, 7))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.5
)

plt.title("EfficientNet-B0 - Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Row-normalized confusion matrix
cm_normalized = cm / np.maximum(
    cm.sum(axis=1, keepdims=True), 1
)

plt.figure(figsize=(9, 7))

sns.heatmap(
    cm_normalized,
    annot=True,
    fmt=".2%",
    cmap="Greens",
    xticklabels=class_names,
    yticklabels=class_names,
    vmin=0,
    vmax=1
)

plt.title("Normalized Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "normalized_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# 12. ROC CURVES
# ============================================================

fpr = {}
tpr = {}
roc_auc = {}

for i in range(num_classes):

    fpr[i], tpr[i], _ = roc_curve(
        y_true_bin[:, i],
        y_prob[:, i]
    )

    roc_auc[i] = auc(
        fpr[i],
        tpr[i]
    )

# Micro-average ROC
fpr["micro"], tpr["micro"], _ = roc_curve(
    y_true_bin.ravel(),
    y_prob.ravel()
)

roc_auc["micro"] = auc(
    fpr["micro"],
    tpr["micro"]
)

# Macro-average ROC
all_fpr = np.unique(
    np.concatenate([
        fpr[i] for i in range(num_classes)
    ])
)

mean_tpr = np.zeros_like(all_fpr)

for i in range(num_classes):
    mean_tpr += np.interp(
        all_fpr,
        fpr[i],
        tpr[i]
    )

mean_tpr /= num_classes

roc_auc["macro"] = auc(
    all_fpr,
    mean_tpr
)

plt.figure(figsize=(10, 8))

for i, class_name in enumerate(class_names):
    plt.plot(
        fpr[i],
        tpr[i],
        linewidth=2,
        label=(
            f"{class_name} (AUC = "
            f"{roc_auc[i]:.4f})"
        )
    )

plt.plot(
    fpr["micro"],
    tpr["micro"],
    linestyle=":",
    linewidth=3,
    label=f"Micro-average (AUC = {roc_auc['micro']:.4f})"
)

plt.plot(
    all_fpr,
    mean_tpr,
    linestyle="--",
    linewidth=3,
    label=f"Macro-average (AUC = {roc_auc['macro']:.4f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    "k--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Multiclass ROC Curves - EfficientNet-B0")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "roc_curves.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# 13. PRECISION-RECALL CURVES
# ============================================================

plt.figure(figsize=(10, 8))

for i, class_name in enumerate(class_names):

    precision_curve, recall_curve, _ = (
        precision_recall_curve(
            y_true_bin[:, i],
            y_prob[:, i]
        )
    )

    class_ap = average_precision_score(
        y_true_bin[:, i],
        y_prob[:, i]
    )

    plt.plot(
        recall_curve,
        precision_curve,
        linewidth=2,
        label=f"{class_name} (AP = {class_ap:.4f})"
    )

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves - EfficientNet-B0")
plt.legend(loc="best")
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "precision_recall_curves.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# 14. PER-CLASS METRIC COMPARISON
# ============================================================

plot_metrics = [
    "Precision",
    "Sensitivity / Recall",
    "Specificity",
    "F1 Score",
    "ROC-AUC (OvR)"
]

plot_df = per_class_df.melt(
    id_vars="Class",
    value_vars=plot_metrics,
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(13, 6))

sns.barplot(
    data=plot_df,
    x="Class",
    y="Score",
    hue="Metric"
)

plt.title("Per-Class Performance Comparison")
plt.ylim(0, 1.05)
plt.xticks(rotation=15)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "per_class_metric_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# 15. CORRECT AND INCORRECT PREDICTIONS
# ============================================================

# Test subset indices correspond to the original ImageFolder
num_to_show = min(16, len(test_indices))

rng = np.random.default_rng(SEED)

candidate_positions = np.arange(len(test_indices))
chosen_positions = rng.choice(
    candidate_positions,
    size=num_to_show,
    replace=False
)

fig, axes = plt.subplots(4, 4, figsize=(14, 13))
axes = axes.flatten()

for plot_idx, position in enumerate(chosen_positions):

    original_index = test_indices[position]

    image_tensor, true_label = eval_dataset_full[
        original_index
    ]

    predicted_label = y_pred[position]
    confidence = float(y_prob[position].max())

    image = denormalize_image(image_tensor)

    axes[plot_idx].imshow(
        image.permute(1, 2, 0)
    )

    correct = true_label == predicted_label

    axes[plot_idx].set_title(
        f"Actual: {class_names[true_label]}\n"
        f"Predicted: {class_names[predicted_label]}\n"
        f"Confidence: {confidence:.2%}",
        color="green" if correct else "red",
        fontsize=9
    )

    axes[plot_idx].axis("off")

for idx in range(num_to_show, len(axes)):
    axes[idx].axis("off")

plt.suptitle(
    "Test Set Predictions (Green = Correct, Red = Incorrect)",
    fontsize=14
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "sample_test_predictions.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# 16. SAVE ALL EVALUATION RESULTS
# ============================================================

# Save test predictions with probabilities
prediction_df = pd.DataFrame({
    "Actual_Class": [class_names[i] for i in y_true],
    "Predicted_Class": [class_names[i] for i in y_pred],
    "Correct": y_true == y_pred,
    "Confidence": y_prob.max(axis=1)
})

for i, class_name in enumerate(class_names):
    prediction_df[f"Probability_{class_name}"] = y_prob[:, i]

prediction_df.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False
)

# Save class names and inference settings
metadata = {
    "model": "EfficientNet-B0",
    "pretrained_weights": "ImageNet",
    "image_size": IMAGE_SIZE,
    "class_names": class_names,
    "seed": SEED,
    "best_epoch": best_epoch,
    "best_validation_loss": float(best_val_loss),
    "training_time_minutes": float(training_time / 60)
}

with open(OUTPUT_DIR / "model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("All results saved to:")
print(OUTPUT_DIR)

print("\nSaved files:")

for file_path in sorted(OUTPUT_DIR.iterdir()):
    print(file_path.name)